# 02 — Recherche sémantique et re-ranking

**Objectif** : valider le retrieval sur 10 requêtes de test.

Ce notebook couvre le point **2.3** du formulaire SpeciTec :
- Mécanisme de recherche (similarité cosine via FAISS)
- Re-ranking (similarity × recency_boost × category_weight)
- Filtrage par métadonnées

In [ ]:
import sys, warnings, logging
from pathlib import Path

warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.retriever import JobRetriever

retriever = JobRetriever()
print(f'Index chargé : {retriever.index.ntotal} documents')

## Test sur 10 requêtes manuelles

In [ ]:
test_queries = [
    "data engineer dbt Geneva",
    "offre power bi lausanne",
    "DBA Oracle suisse romande",
    "ETL developer sans cloud",
    "BI analyst tableau reporting",
    "support applicatif SAP",
    "data engineer junior sans expérience",
    "postes récents data engineering",
    "offres entreprises bancaires genève",
    "comparaison offres python vs SQL"
]

for query in test_queries:
    print(f'\n=== "{query}" ===')
    results = retriever.search(query, top_k=3)
    if not results:
        print('  → Aucun résultat au-dessus du seuil de similarité')
    for r in results:
        print(f"  [{r['rank']}] score={r['score']:.4f} sim={r['similarity']:.4f}")
        print(f"       {r['title'][:60]} — {r['company'][:30]}")
        print(f"       {r['location'][:40]} | {r['label']} | {r['date_posted']}")

## Test du filtrage par métadonnées

In [ ]:
# Filtre par catégorie
print('=== Filtre : DATA_ENGINEERING uniquement ===')
results = retriever.search(
    "ingénieur données pipeline",
    filters={"label": "DATA_ENGINEERING"}
)
for r in results:
    print(f"  [{r['rank']}] {r['score']:.4f} | {r['title'][:55]} | {r['label']}")

print()

# Filtre par localisation
print('=== Filtre : location contient "Genève" ===')
results = retriever.search(
    "data analyst",
    filters={"location_contains": "Genève"}
)
for r in results:
    print(f"  [{r['rank']}] {r['score']:.4f} | {r['title'][:50]} | {r['location'][:30]}")

## Analyse du re-ranking : avant vs après

In [ ]:
import numpy as np

# Comparaison : tri par similarité brute vs score re-ranké
query = "data engineer pipeline Zurich"
results = retriever.search(query, top_k=10)

print(f'Requête : "{query}"')
print()
print(f'{"Rang":<5} {"Score":<8} {"Sim":<8} {"Recency":<8} {"Titre":<45} {"Date"}')
print('-' * 100)
for r in results:
    print(f"{r['rank']:<5} {r['score']:<8.4f} {r['similarity']:<8.4f} {r['recency_boost']:<8.3f} {r['title'][:44]:<45} {r['date_posted']}")

---

## Récapitulatif Phase 2

| Paramètre | Valeur |
|-----------|--------|
| Candidats FAISS | 20 (TOP_K_RETRIEVAL) |
| Résultats finaux | 5 (TOP_K_FINAL) |
| Seuil minimal | 0.30 (MIN_SIMILARITY) |
| Re-ranking | sim × recency × category_weight |
| Demi-vie recency | 30 jours |

**Prochaine étape** : `03_rag_pipeline.ipynb` — connexion LLM Claude pour génération de réponses